# 02 — Precificação, Curva de Juros (ETTJ) e Marcação a Mercado

Avaliação integrada de títulos nominais e indexados ao IPCA (NTN-B) pelo padrão
STN/ANBIMA (base 252, calendário B3/SFN e liquidação D+1), cálculo de YTM
implícita e marcação a mercado acoplada à Estrutura a Termo da Taxa de Juros (`ETTJ`).

In [1]:
from datetime import date

from rfquant.pricing import marktomarket, precificar, ytm
from rfquant.pricing.curva_juros import ETTJ

# Parâmetros base (Tesouro Prefixado com Juros Semestrais / NTN-F ou NTN-B)
data_base = date(2026, 1, 2)
vencimento = date(2035, 5, 15)
taxa = 0.062  # 6,2% a.a.
cupom = 0.06  # 6,0% a.a. (cupom semestral composto STN/ANBIMA)
valor_nominal = 1000.0

In [2]:
# 1) Avaliação unificada de título nominal (PU, durations, DV01, convexidade e Current Yield)
av = precificar(
    data_base=data_base,
    data_vencimento=vencimento,
    taxa_anual=taxa,
    valor_nominal=valor_nominal,
    taxa_cupom_anual=cupom,
)

print(f"PU:                  R$ {av.pu:,.2f}")
print(f"Duration Macaulay:   {av.duration_macaulay:.4f} anos")
print(f"Duration Modificada: {av.duration_modificada:.4f}")
print(f"DV01 (+1 bp):        R$ {av.dv01:.4f} / bp")
print(f"Convexidade:         {av.convexidade:.4f}")
print(f"Current Yield:       {av.current_yield:.4%}")

PU:                  R$ 997.39
Duration Macaulay:   7.2002 anos
Duration Modificada: 6.7799
DV01 (+1 bp):        R$ 0.6759 / bp
Convexidade:         60.3486
Current Yield:       5.9281%


In [3]:
# 2) Modo IPCA+ (NTN-B): cotação em % do VNA (base 100) e PU escalado pelo VNA do dia
av_ntnb = precificar(
    data_base=data_base,
    data_vencimento=vencimento,
    taxa_anual=taxa,
    taxa_cupom_anual=cupom,
    vna=4500.0,
)

print(f"Cotação NTN-B:       {av_ntnb.cotacao:.4f} % do VNA")
print(f"PU NTN-B (VNA 4500): R$ {av_ntnb.pu:,.2f}")
print(f"DV01 NTN-B:          R$ {av_ntnb.dv01:.4f} / bp")
print(f"Current Yield NTN-B: {av_ntnb.current_yield:.4%}")

Cotação NTN-B:       99.7392 % do VNA
PU NTN-B (VNA 4500): R$ 4,488.26
DV01 NTN-B:          R$ 3.0416 / bp
Current Yield NTN-B: 5.9281%


In [4]:
# 3) YTM: dado um PU de mercado com deságio (R$ 980,00 sobre R$ 1.000,00), qual a taxa implícita?
pu_mercado = 980.0
y = ytm.calcular(pu_mercado, data_base, vencimento, valor_nominal, cupom)
print(f"YTM implícita no PU R$ {pu_mercado:,.2f}: {y:.4%} a.a.")

YTM implícita no PU R$ 980.00: 6.4602% a.a.


In [5]:
# 4) Marcação a mercado exata acoplada à curva de juros (ETTJ flat-forward B3/ANBIMA)
curva_ettj = ETTJ(
    pontos=[(1.0, 0.055), (3.0, 0.057), (5.0, 0.058), (10.0, 0.060)],
    metodo="flat_forward",
)

res = marktomarket.exata(
    data_compra=data_base,
    data_hoje=date(2026, 6, 1),
    data_vencimento=vencimento,
    taxa_compra=taxa,
    valor_nominal=valor_nominal,
    taxa_cupom_anual=cupom,
    ettj=curva_ettj,
)

print(f"Taxa interpolada na ETTJ: {res.taxa_mercado:.4%} a.a.")
print(f"PU na compra:             R$ {res.pu_compra:,.2f}")
print(f"PU na curva hoje:         R$ {res.pu_curva:,.2f}")
print(f"PU a mercado hoje:        R$ {res.pu_mercado:,.2f}")
print(f"Variação MtM:             R$ {res.variacao_absoluta:+,.2f} ({res.variacao_percentual:+.4%})")

Taxa interpolada na ETTJ: 5.9752% a.a.
PU na compra:             R$ 997.39
PU na curva hoje:         R$ 992.09
PU a mercado hoje:        R$ 1,006.94
Variação MtM:             R$ +14.85 (+1.4971%)
